# Setup and data loading

In [1]:
import pandas as pd
from linearmodels.panel import PanelOLS  # The 'Gold Standard' for Panel Regressions

# Loading the prepped panel dataset
df_panel = pd.read_csv("../Data/prepared_panel_data.csv")

# Creation of a multindex since we do have two dimensions to control for
panel_data = df_panel.set_index(["Vessel Name", "TimeIdx"], drop=False)

# These are our dependent variables. We will run a separate regression for each to see the policy's
# multi-dimensional impact
outcomes = [
    "log_CO2_per_nm", # Environmental impact
    "MGO_Share", # Fuel switching (clean fuel)
    "HSFO_Share", # Fuel switching (dirty fuel)
    "VLSFO_Share", # Fuel switching (standard fuel)
    "Avg Speed (kn)", # Operational response
    "Distance (nm)"] # Routing/evasion response

print("Total observations for the analysis:", len(panel_data))
print("Total unique vessels:", panel_data['Vessel Name'].nunique())

Total observations for the analysis: 1755789
Total unique vessels: 15781


# Global comparative Analysis (Baseline vs. Trend)

In [2]:
# We iterate through every outcome (CO2, Speed, Fuel, etc.) and run twodifferent model
# specifications to see how robust the policy effect is to time trends from the previous year

# Print a formatted header for the results table
print(f"{'Outcome Variable':<25} | {'Baseline (No Trend)':<22} | {'Main Spec (With Trend)':<22}")
print("-" * 75)

for outcome in outcomes:
    # Model A : the baseline specification (no pre-trend control)
    # This is a standard Difference-in-Differences (DiD) model; it assumes that if the tax
    # hadn't happened, the treated and control groups would have followed exactly parallel paths
    exog_baseline = ["Interaction", "ETS_Rate"]
    
    res_base = PanelOLS(
        dependent=panel_data[outcome],
        exog=panel_data[exog_baseline],
        entity_effects=True, # Subtracts individual ship quirks
        time_effects=True # Subtracts global monthly shocks
    ).fit(cov_type="clustered", cluster_entity=True, low_memory=True)
    
    # Export the full technical output (R-squared, F-stats, etc.) to a text file, for documentation
    with open(f"../Results/Results_regressions/GLOBAL_{outcome}_Baseline.txt", "w") as f:
        f.write(res_base.summary.as_text())
        
    # Extract the interaction coefficient and p-value
    b_coef = res_base.params["Interaction"]
    b_pval = res_base.pvalues["Interaction"]
    b_star = "***" if b_pval < 0.01 else ("**" if b_pval < 0.05 else ("*" if b_pval < 0.10 else ""))

    # Model B : the trend control specification (which is our main specification)
    # This model is more conservative and rigorous than the baseline. It includes 'ETS_x_Time'
    # (Treatment Intensity * Continuous Time Index). Actually, tt relaxes the parallel trends
    # assumption. It accounts for the possibility that EU-exposed ships had different slopes of behavior 
    # (e.g., were already slowing down) before the policy started. By soaking up the long-term
    # linear trend with ETS_x_Time, the interaction coefficient is forced to isolate only the
    # discontinuity or the sudden jump in behavior that happened in 2024
    exog_trend = ["Interaction", "ETS_Rate", "ETS_x_Time"]
    
    res_trend = PanelOLS(
        dependent=panel_data[outcome],
        exog=panel_data[exog_trend],
        entity_effects=True,
        time_effects=True
    ).fit(cov_type="clustered", cluster_entity=True, low_memory=True)
    
    # Export results for documentation
    with open(f"../Results/Results_regressions/GLOBAL_{outcome}_Trend.txt", "w") as f:
        f.write(res_trend.summary.as_text())
        
    # Extract results for the summary table
    t_coef = res_trend.params["Interaction"]
    t_pval = res_trend.pvalues["Interaction"]
    t_star = "***" if t_pval < 0.01 else ("**" if t_pval < 0.05 else ("*" if t_pval < 0.10 else ""))
    
    # Print the comparison row
    print(f"{outcome:<25} | {b_coef:>8.4f} (p={b_pval:.3f}){b_star:<3} | {t_coef:>8.4f} (p={t_pval:.3f}){t_star:<3}")

Outcome Variable          | Baseline (No Trend)    | Main Spec (With Trend)
---------------------------------------------------------------------------
log_CO2_per_nm            |   0.0167 (p=0.000)*** |   0.0039 (p=0.020)** 
MGO_Share                 |   0.0005 (p=0.010)*** |  -0.0001 (p=0.667)   
HSFO_Share                |  -0.0002 (p=0.106)    |   0.0001 (p=0.109)   
VLSFO_Share               |  -0.0004 (p=0.126)    |  -0.0000 (p=0.939)   
Avg Speed (kn)            |   0.1796 (p=0.000)*** |   0.0102 (p=0.448)   
Distance (nm)             | 286.5516 (p=0.000)*** |  57.5665 (p=0.048)** 


# Strategic response by trade profile (cluster analysis)

In [3]:
# Why do we do this? A global average can be misleading. For example, if some ships slow down
# and others speed up, the global average might show no change. By splitting the data into our
# pre-defined clusters, we can see the unique strategic signatures of the different trade groups

# These are the three profiles identified by our ML step
clusters_to_test = ["Low EU", "Med EU", "High EU"]

# We test all 6 outcomes to see the full operational shift (Speed, Fuel, and Emissions)
outcomes_to_test = ["log_CO2_per_nm", "MGO_Share", "HSFO_Share", "VLSFO_Share", "Avg Speed (kn)", "Distance (nm)"]

for cluster in clusters_to_test:
    # We isolate only the vessels belonging to one specific trade profile.
    cluster_data = panel_data[panel_data["Cluster_Name"] == cluster]
    
    print(f"Cluster: {cluster} (Strategic Profile) ---")
    
    for outcome in outcomes_to_test:
        # We directly use the main specification (model B) with trend control
        exog_trend = ["Interaction", "ETS_Rate", "ETS_x_Time"]
        
        res_cluster = PanelOLS(
            dependent=cluster_data[outcome],
            exog=cluster_data[exog_trend],
            entity_effects=True, # Controls for vessel-specific 'DNA'
            time_effects=True # Controls for global monthly shocks
        ).fit(cov_type="clustered", cluster_entity=True, low_memory=True)
        
        # We export the full technical summary for each test for documentation too
        file_name = f"../Results/Results_regressions/CLUSTER_{cluster.replace(' ', '_')}_{outcome}.txt"
        with open(file_name, "w") as f:
            f.write(res_cluster.summary.as_text())
            
        coef = res_cluster.params["Interaction"]
        pval = res_cluster.pvalues["Interaction"]
        star = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.10 else ""))
        
        # Immediate console output for real-time monitoring
        print(f"   -> {outcome:<18}: {coef:>8.4f} (p={pval:.4f}) {star}")
        
    print("")

Cluster: Low EU (Strategic Profile) ---
   -> log_CO2_per_nm    :  -0.0227 (p=0.0145) **
   -> MGO_Share         :  -0.0005 (p=0.7043) 
   -> HSFO_Share        :   0.0001 (p=0.8239) 
   -> VLSFO_Share       :   0.0005 (p=0.7336) 
   -> Avg Speed (kn)    :  -0.1576 (p=0.0011) ***
   -> Distance (nm)     : 772.9628 (p=0.0000) ***

Cluster: Med EU (Strategic Profile) ---
   -> log_CO2_per_nm    :  -0.0681 (p=0.0000) ***
   -> MGO_Share         :  -0.0003 (p=0.6237) 
   -> HSFO_Share        :  -0.0012 (p=0.0338) **
   -> VLSFO_Share       :   0.0015 (p=0.0345) **
   -> Avg Speed (kn)    :  -0.3039 (p=0.0000) ***
   -> Distance (nm)     : 362.9412 (p=0.0516) *

Cluster: High EU (Strategic Profile) ---
   -> log_CO2_per_nm    :   0.0404 (p=0.0000) ***
   -> MGO_Share         :   0.0037 (p=0.0000) ***
   -> HSFO_Share        :  -0.0000 (p=0.2002) 
   -> VLSFO_Share       :  -0.0037 (p=0.0000) ***
   -> Avg Speed (kn)    :  -0.1141 (p=0.0000) ***
   -> Distance (nm)     : -413.0376 (p=0.0000) 

# Robustness check I : excluding containers

In [4]:
# In this dataset, general cargo and project cargo are the sectors most likely to behave differently
# from the heavy industrial commodities (like iron ore or coal). We exclude them to see if
# the policy effect holds for the pure bulk fleet

# Storing the non bulk sectors
exclusion_keywords = ["General Cargo", "Project Cargo", "Unknown"]

# We filter out ships that primarily carry general cargo or project cargo
non_specialized_mask = panel_data["Primary_Commodity"].isin(exclusion_keywords)
non_container_data = panel_data[~non_specialized_mask].copy()

print("Vessels in the full dataset:", panel_data['Vessel Name'].nunique())
print("Vessels in the pure bulk sample:", non_container_data['Vessel Name'].nunique())

outcomes_to_test = ["log_CO2_per_nm", "Avg Speed (kn)"]

for outcome in outcomes_to_test:
    # We maintain the main spec
    exog_trend = ["Interaction", "ETS_Rate", "ETS_x_Time"]
    
    res_robust = PanelOLS(
        dependent=non_container_data[outcome],
        exog=non_container_data[exog_trend],
        entity_effects=True,
        time_effects=True
    ).fit(cov_type="clustered", cluster_entity=True, low_memory=True)
    
    # Save the summary
    file_path = f"../Results/Results_regressions/ROBUST_PureBulk_{outcome}.txt"
    with open(file_path, "w") as f:
        f.write(res_robust.summary.as_text())
        
    coef = res_robust.params["Interaction"]
    pval = res_robust.pvalues["Interaction"]
    star = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.10 else ""))
    
    print(f"   -> {outcome:<18}: {coef:>8.4f} (p={pval:.4f}) {star}")

Vessels in the full dataset: 15781
Vessels in the pure bulk sample: 15509
   -> log_CO2_per_nm    :   0.0040 (p=0.0198) **
   -> Avg Speed (kn)    :   0.0096 (p=0.4777) 
